<div align="right">
    <img src="./Imagenes/logo_UTN.svg" alt="Logo de la UTN" width="150" />
</div>

#### Teoría de los Circuitos II
# Trabajo Semanal 8
#### Alumno: Lucas Gallero
#### Profesor: Mariano Llamedo Soria
#### Ayudante de TPs: David Moharos
----

# Enunciado

### 1) Síntesis de un dipolo RLC

Dada una función impedancia $Z(s)$, se sabe que la pulsación libre de amortiguamiento y el coeficiente de amortiguamiento asociados a sus ceros son $1\;\mathrm{rad/s}$ y $0{,}5\;\mathrm{s^{-1}}$. Posee polos en $s=-1$ y $s=5e^{j116{,}6^\circ}$.

Determine el valor de los componentes del siguiente dipolo, sabiendo que satisface la impedancia propuesta.

<div align="center">
    <img src="./Imagenes/Enunciado1.png" alt="Dipolo del ejercicio 1" width="550" />
</div>

### 2) Síntesis de un dipolo RC con constantes de tiempo dadas

Encuentre los elementos del siguiente dipolo, caracterizado por:

$$R_1C_1=\frac15\;\mathrm{s},\qquad R_2C_2=\frac14\;\mathrm{s},$$

$$Z(s)=\frac{s^2+6s+8}{s^2+4s+3}.$$

<div align="center">
    <img src="./Imagenes/Enunciado2.png" alt="Topología propuesta para el ejercicio 2" width="400" />
</div>

En ambos ejercicios se pide verificar mediante simulación simbólica en Python con PyTC2 y simulación circuital en LTspice, utilizando cursores para verificar las singularidades.


In [1]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Math, HTML
from io import BytesIO
import base64
from pytc2.general import s, print_latex, a_equal_b_latex_s
from pytc2.remociones import (
    remover_polo_infinito,
    remover_valor_en_infinito,
    remover_polo_sigma,
)

%matplotlib inline
sp.init_printing()
plt.rcParams.update({'font.size': 11, 'figure.dpi': 100})

def mostrar(nombre, expresion):
    # Los valores se calculan sin redondear; solo se abrevia su presentación.
    if any(max(abs(r.p),abs(r.q))>10**6 for r in expresion.atoms(sp.Rational)):
        num, den = sp.fraction(sp.cancel(expresion))
        escala = sp.Poly(den,s).LC()
        num = sp.N(num/escala,7)
        den = sp.N(den/escala,7)
        vista = num if den.equals(1) else sp.Mul(num,sp.Pow(den,-1,evaluate=False),evaluate=False)
        display(Math(nombre+r'\approx '+sp.latex(vista)))
    else:
        print_latex(a_equal_b_latex_s(nombre, sp.factor(expresion)))

def tabla_componentes(componentes):
    for nombre, valor, unidad in componentes:
        exacto = f'{sp.sstr(valor)} = ' if all(abs(r.q)<10**6 for r in valor.atoms(sp.Rational)) else ''
        print(f'{nombre}: {exacto}{float(valor):.9g} {unidad}')

def verificar_cero(expresion):
    assert sp.cancel(expresion) == 0

def mostrar_figura(figura, nombre, titulo, ancho=1000):
    figura.savefig('Imagenes/'+nombre, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(figura)
    display(HTML(f'<div align="center">\n    <img src="./Imagenes/{nombre}" alt="{titulo}" width="{ancho}" />\n</div>'))

def bode(funcion, titulo, limites=(1e-3, 1e2)):
    f = np.logspace(np.log10(limites[0]), np.log10(limites[1]), 3000)
    z = sp.lambdify(s, funcion, 'numpy')(2j*np.pi*f)
    fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)
    ax[0].semilogx(f, 20*np.log10(np.abs(z)))
    ax[0].set_ylabel('Módulo de Z (dB ref. 1 Ω)')
    ax[0].set_title(titulo)
    ax[1].semilogx(f, np.angle(z, deg=True))
    ax[1].set_ylabel('Fase de Z (°)')
    ax[1].set_xlabel('Frecuencia (Hz)')
    for eje in ax:
        eje.grid(True, which='both', alpha=.3)
    nombre = "TS8_bode_1.png" if "Ejercicio 1" in titulo else "TS8_bode_2.png"
    mostrar_figura(fig, nombre, titulo)

---
# <ins>Resolución</ins>

## <ins>1. Síntesis del dipolo RLC</ins>

Se sigue el orden del desarrollo manuscrito: capacitor en derivación, resistencia en derivación, inductancia serie, resistencia serie y paralelo RC final. Se conserva la numeración de los remanentes $Y_2$, $Y_4$, $Z_4$, $Z_6$ y $Z_8$.

### <ins>Ceros, polos y función impedancia con K=1</ins>

Con $\omega_0=1$ y $\alpha=0{,}5$, los ceros son:

$$z_{1,2}=-\alpha\pm j\sqrt{\omega_0^2-\alpha^2}
=-\frac12\pm j\frac{\sqrt3}{2}.$$

Su polinomio es:

$$N(s)=s^2+2\alpha s+\omega_0^2=s^2+s+1.$$

El polo complejo se pasa a forma rectangular:

$$p_{2,3}=5\cos116{,}6^\circ\pm j\,5\sin116{,}6^\circ
\approx-2{,}238795\pm j\,4{,}470771.$$

Se incluye el conjugado porque el circuito tiene coeficientes reales. Junto con $p_1=-1$:

$$D(s)=(s+1)(s^2+hs+25),\qquad h=-10\cos116{,}6^\circ\approx4{,}477590878.$$

Los polos y ceros no fijan por sí solos la escala de impedancia. Se adopta **$K=1$**

$$\boxed{Z(s)=\frac{s^2+s+1}{(s+1)(s^2+hs+25)}}.$$

$$Z(s)\approx\frac{s^2+s+1}{s^3+5{,}477590878s^2+29{,}477590878s+25}.$$

En las remociones se conserva $h$ sin redondearlo a tres decimales. Esto evita acumular diferencias en los componentes.

In [2]:
omega0 = sp.Integer(1)
alpha = sp.Rational(1, 2)
angulo = sp.Rational(583, 5)*sp.pi/180
h_angular = -10*sp.cos(angulo)
# Se conserva el coeficiente angular con 16 cifras y se opera con racionales.
h = sp.Rational(str(sp.N(h_angular, 16)))
Z_ej1 = (s**2+s+1)/((s+1)*(s**2+h*s+25))
mostrar('Z(s)', Z_ej1)
ceros_ej1 = [-alpha-sp.I*sp.sqrt(omega0**2-alpha**2),
             -alpha+sp.I*sp.sqrt(omega0**2-alpha**2)]
polos_ej1 = [sp.Integer(-1), 5*sp.exp(sp.I*angulo), 5*sp.exp(-sp.I*angulo)]
print('Ceros:', [complex(sp.N(z)) for z in ceros_ej1])
print('Polos:', [complex(sp.N(p)) for p in polos_ej1])
print('h =', float(h))

<IPython.core.display.Math object>

Ceros: [(-0.5-0.8660254037844386j), (-0.5+0.8660254037844386j)]
Polos: [(-1+0j), (-2.2387954391938485+4.470771184196841j), (-2.2387954391938485-4.470771184196841j)]
h = 4.477590878387697


### <ins>Cómo se interpretan las remociones</ins>

Una remoción resta el aporte de un elemento de la función de entrada. Para elementos en paralelo se trabaja con admitancia; para elementos en serie, con impedancia.

| Elemento | Término removido | Función utilizada |
|---|---|---|
| Capacitor en derivación | $sC$ | $Y$ |
| Resistor en derivación | $1/R$ | $Y$ |
| Inductor serie | $sL$ | $Z$ |
| Resistor serie | $R$ | $Z$ |
| Paralelo RC en serie con el resto | $k/(s+\sigma)$ | $Z$ |

La red completa es RLC. Las primeras extracciones son remociones en infinito; el último paralelo se obtiene con una remoción Foster RC de polo real.

### <ins>Primera remoción: capacitor C1 en derivación</ins>

Se invierte la impedancia porque $C_1$ está en paralelo con la entrada:

$$Y(s)=\frac1{Z(s)}=\frac{s^3+(h+1)s^2+(h+25)s+25}{s^2+s+1}.$$

La remoción del polo en infinito da:

$$C_1=\lim_{s\to\infty}\frac{Y(s)}s=1\;\mathrm{F}.$$

Se resta $Y_{C_1}=s$:

$$\begin{aligned}
Y_2&=Y-s\\
&=\frac{s^3+(h+1)s^2+(h+25)s+25-s(s^2+s+1)}{s^2+s+1}\\
&=\boxed{\frac{hs^2+(h+24)s+25}{s^2+s+1}}.
\end{aligned}$$

In [3]:
Y_ej1 = sp.cancel(1/Z_ej1)
Y2_ej1, Y_C1 = remover_polo_infinito(Y_ej1)
C1_ej1 = sp.simplify(Y_C1/s)
mostrar('Y_{C_1}(s)', Y_C1)
mostrar('Y_2(s)', Y2_ej1)
mostrar('C_1', C1_ej1)
verificar_cero(Y_ej1-Y_C1-Y2_ej1)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### <ins>Segunda remoción: resistencia R1 en derivación</ins>

Después de remover $C_1$, a alta frecuencia la inductancia abre la rama derecha. Queda la conductancia de $R_1$:

$$G_1=\frac1{R_1}=\lim_{s\to\infty}Y_2(s)=h.$$

$$\boxed{R_1=\frac1h\approx0{,}223334\;\Omega}.$$

Se remueve el valor constante:

$$\begin{aligned}
Y_4&=Y_2-h\\
&=\frac{hs^2+(h+24)s+25-h(s^2+s+1)}{s^2+s+1}\\
&=\boxed{\frac{24s+25-h}{s^2+s+1}}.
\end{aligned}$$

Solo queda la rama $R_2+L_1+(R_3\parallel C_2)$. Se invierte su admitancia para extraer los elementos serie:

$$\boxed{Z_4=\frac1{Y_4}=\frac{s^2+s+1}{24s+25-h}}.$$

Para simplificar las siguientes cuentas se define:

$$\sigma=\frac{25-h}{24}\approx0{,}855100380,
\qquad Z_4=\frac{s^2+s+1}{24(s+\sigma)}.$$

In [4]:
Y4_ej1, G1_ej1 = remover_valor_en_infinito(Y2_ej1)
R1_ej1 = sp.simplify(1/G1_ej1)
Z4_ej1 = sp.cancel(1/Y4_ej1)
sigma_ej1 = sp.simplify((25-h)/24)
mostrar('G_1', G1_ej1)
mostrar('Y_4(s)', Y4_ej1)
mostrar('Z_4(s)', Z4_ej1)
tabla_componentes([('R1', R1_ej1, 'Ω')])
verificar_cero(Y2_ej1-G1_ej1-Y4_ej1)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

R1: 0.223334384 Ω


### <ins>Tercera remoción: inductancia L1 serie</ins>

La remoción del polo en infinito de la impedancia permite extraer $sL_1$:

$$L_1=\lim_{s\to\infty}\frac{Z_4(s)}s
=\lim_{s\to\infty}\frac{s^2+s+1}{24s(s+\sigma)}
=\boxed{\frac1{24}\;\mathrm{H}}.$$

$$\begin{aligned}
Z_6&=Z_4-\frac{s}{24}\\
&=\frac{s^2+s+1-s(s+\sigma)}{24(s+\sigma)}\\
&=\boxed{\frac{(1-\sigma)s+1}{24(s+\sigma)}}.
\end{aligned}$$

### <ins>Cuarta remoción: resistencia R2 serie</ins>

A alta frecuencia el capacitor de salida cortocircuita su paralelo. Por lo tanto:

$$R_2=\lim_{s\to\infty}Z_6(s)
=\frac{1-\sigma}{24}
\approx\boxed{0{,}00603748\;\Omega}.$$

$$\begin{aligned}
Z_8&=Z_6-R_2\\
&=\frac{(1-\sigma)s+1-(1-\sigma)(s+\sigma)}{24(s+\sigma)}\\
&=\boxed{\frac{1-\sigma+\sigma^2}{24(s+\sigma)}}.
\end{aligned}$$

Los dos términos extraídos forman el bloque RL serie $Z_{RL}=R_2+sL_1$.

In [5]:
Z6_ej1, Z_L1 = remover_polo_infinito(Z4_ej1)
L1_ej1 = sp.simplify(Z_L1/s)
Z8_ej1, R2_ej1 = remover_valor_en_infinito(Z6_ej1)
mostrar('Z_{L_1}(s)', Z_L1)
mostrar('Z_6(s)', Z6_ej1)
mostrar('Z_8(s)', Z8_ej1)
tabla_componentes([('L1', L1_ej1, 'H'), ('R2', R2_ej1, 'Ω')])
verificar_cero(Z4_ej1-Z_L1-Z6_ej1)
verificar_cero(Z6_ej1-R2_ej1-Z8_ej1)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

L1: 1/24 = 0.0416666667 H
R2: 0.00603748416 Ω


### <ins>Quinta remoción: Foster RC del paralelo final</ins>

El remanente tiene un único polo real en $s=-\sigma$:

$$Z_8(s)=\frac{k_\sigma}{s+\sigma},\qquad
k_\sigma=\lim_{s\to-\sigma}(s+\sigma)Z_8(s)
=\frac{1-\sigma+\sigma^2}{24}\approx0{,}03650401166.$$

$k_\sigma$ es el residuo de esta remoción; no es la constante $K=1$ fijada al construir $Z(s)$.

La impedancia de un paralelo RC es:

$$Z_{R_3\parallel C_2}
=\frac1{sC_2+1/R_3}
=\frac{1/C_2}{s+1/(R_3C_2)}.$$

Comparando con la forma Foster:

$$\frac1{C_2}=k_\sigma,\qquad\frac1{R_3C_2}=\sigma.$$

$$\boxed{C_2=\frac1{k_\sigma}\approx27{,}39424941\;\mathrm{F}},
\qquad
\boxed{R_3=\frac{k_\sigma}{\sigma}\approx0{,}04268973856\;\Omega}.$$

Al remover todo el paralelo, $Z_{10}=Z_8-k_\sigma/(s+\sigma)=0$.

En PyTC2 se utiliza `isImpedance=True, isRC=True`: el término removido es una **impedancia RC** que se conecta en serie con el resto.

In [6]:
Z10_ej1, Z_RC_final, R3_ej1, C2_ej1 = remover_polo_sigma(
    Z8_ej1, sigma=sigma_ej1, isImpedance=True, isRC=True
)
k_sigma_ej1 = sp.simplify((1-sigma_ej1+sigma_ej1**2)/24)
print('sigma =', float(sigma_ej1), 's^-1')
print('k_sigma =', float(k_sigma_ej1))
tabla_componentes([('R3', R3_ej1, 'Ω'), ('C2', C2_ej1, 'F')])
mostrar('Z_{10}(s)', Z10_ej1)
verificar_cero(Z10_ej1)
verificar_cero(Z_RC_final-1/(s*C2_ej1+1/R3_ej1))

sigma = 0.8551003800671793 s^-1
k_sigma = 0.036504011663493964
R3: 0.0426897386 Ω
C2: 27.3942494 F


<IPython.core.display.Math object>

### <ins>Componentes y circuito obtenido</ins>

| Componente | Valor | Conexión |
|---|---:|---|
| $C_1$ | $1\;\mathrm{F}$ | Derivación en la entrada |
| $R_1$ | $0{,}223334\;\Omega$ | Derivación en la entrada |
| $R_2$ | $0{,}00603748\;\Omega$ | Serie |
| $L_1$ | $1/24\;\mathrm{H}$ | Serie |
| $C_2$ | $27{,}39424941\;\mathrm{F}$ | Paralelo final |
| $R_3$ | $0{,}04268973856\;\Omega$ | Paralelo final |

<div align="center">
    <img src="./Imagenes/TS8_circuito_1.png" alt="Circuito obtenido para K igual a 1" width="1000" />
</div>

### <ins>Reconstrucción y verificación simbólica</ins>

Se reconstruye desde el paralelo de salida hacia la entrada:

$$Z_{\mathrm{salida}}=\frac1{sC_2+1/R_3},\qquad
Z_{\mathrm{rama}}=R_2+sL_1+Z_{\mathrm{salida}}.$$

$$Y_{\mathrm{red}}=sC_1+\frac1{R_1}+\frac1{Z_{\mathrm{rama}}},
\qquad Z_{\mathrm{red}}=\frac1{Y_{\mathrm{red}}}.$$

La diferencia entre la función reconstruida y la propuesta debe ser idénticamente cero.

In [7]:
Z_salida_ej1 = 1/(s*C2_ej1+1/R3_ej1)
Z_rama_ej1 = R2_ej1+s*L1_ej1+Z_salida_ej1
Z_red_ej1 = sp.cancel(1/(s*C1_ej1+1/R1_ej1+1/Z_rama_ej1))
diferencia_ej1 = sp.cancel(Z_red_ej1-Z_ej1)
mostrar('Z_{red}(s)-Z(s)', diferencia_ej1)
verificar_cero(diferencia_ej1)
assert all(float(x)>0 for x in [C1_ej1,R1_ej1,R2_ej1,L1_ej1,C2_ej1,R3_ej1])

<IPython.core.display.Math object>

### <ins>Comportamiento en baja y alta frecuencia</ins>

En DC, los capacitores quedan abiertos y la inductancia en cortocircuito:

$$Z(0)=R_1\parallel(R_2+R_3)=\frac1{25}=0{,}04\;\Omega.$$

A alta frecuencia, el capacitor de entrada domina:

$$Z(s)\sim\frac1{sC_1}=\frac1s.$$

El módulo tiende a $-20\;\mathrm{dB/década}$ y la fase a $-90^\circ$. En baja frecuencia el módulo tiende a $20\log_{10}(0{,}04)=-27{,}9588\;\mathrm{dB}$ y la fase a $0^\circ$.

Los ceros y polos complejos están en el semiplano izquierdo, fuera del eje $j\omega$. Por eso no producen nulos exactos ni divergencias en el barrido AC.

In [8]:
Z_dc_ej1 = sp.simplify(R1_ej1*(R2_ej1+R3_ej1)/(R1_ej1+R2_ej1+R3_ej1))
mostrar('Z(0)', Z_dc_ej1)
assert Z_dc_ej1 == sp.Rational(1,25)
bode(Z_ej1, 'Ejercicio 1 - Impedancia teórica, K=1')

<IPython.core.display.Math object>

### <ins>Simulación en LTspice</ins>

**Circuito simulado**

<div align="center">
    <img src="./Imagenes/Ejercicio1 - Simulacion - Circuito .png" alt="Circuito del ejercicio 1 en LTspice" width="1000" />
</div>

La impedancia se mide con `V(in)/(-I(V1))`. El signo negativo toma la corriente que entra al dipolo. La inductancia tiene `Rser=0`, de modo que corresponde al componente ideal utilizado en la síntesis.

**Módulo y fase**

<div align="center">
    <img src="./Imagenes/Ejercicio1 - Simulacion.png" alt="Módulo y fase de la impedancia del ejercicio 1" width="1000" />
</div>

En baja frecuencia el módulo se aproxima a $-27{,}96\;\mathrm{dB}$, correspondiente a $Z(0)=0{,}04\;\Omega$, y la fase está próxima a $0^\circ$. Se observan un mínimo y un máximo finitos; a alta frecuencia el módulo decrece y la fase tiende a $-90^\circ$.

Los polos y ceros complejos están fuera del eje $j\omega$. Sus posiciones se verifican mediante las raíces calculadas en la resolución simbólica; los extremos del módulo no coinciden, en general, con las partes imaginarias ni con los módulos de esas raíces.

**Cursores en baja frecuencia**

<div align="center">
    <img src="./Imagenes/Ejercicio1 - Simulacion - Cursores Baja Freq.png" alt="Cursores de baja frecuencia del ejercicio 1" width="300" />
</div>

La captura contiene dos lecturas: $f_1=1\;\mathrm{mHz}$ y $f_2=10\;\mathrm{mHz}$, con módulos $M_1=-27{,}959135\;\mathrm{dB}$ y $M_2=-27{,}992201\;\mathrm{dB}$. Las fases son $-0{,}064463171^\circ$ y $-0{,}63063546^\circ$, respectivamente.

LTspice muestra una diferencia de módulo de $-33{,}066457\;\mathrm{mdB}=-0{,}033066457\;\mathrm{dB}$. Como los cursores están separados una década:

$$m_{\mathrm{baja}}=\frac{-0{,}033066457}{\log_{10}(0{,}01/0{,}001)}
=-0{,}033066457\;\mathrm{dB/década}\approx-0{,}03307\;\mathrm{dB/década}.$$

El resultado está próximo a $0\;\mathrm{dB/década}$, como corresponde a la impedancia resistiva $Z(0)=0{,}04\;\Omega$. Las fases pequeñas también son consistentes con ese comportamiento.

**Cursores en alta frecuencia**

<div align="center">
    <img src="./Imagenes/Ejercicio1 - Simulacion - Cursores Alta Freq.png" alt="Cursores de alta frecuencia del ejercicio 1" width="300" />
</div>

La pendiente se calcula a partir de la diferencia de módulo y de la separación logarítmica de las frecuencias:

$$m=\frac{M_2-M_1}{\log_{10}(f_2/f_1)}\quad[\mathrm{dB/década}].$$

| Zona | $f_1$ (Hz) | $f_2$ (Hz) | $M_1$ (dB) | $M_2$ (dB) | Pendiente (dB/década) |
|---|---:|---:|---:|---:|---:|
| Baja frecuencia | $0{,}001$ | $0{,}01$ | $-27{,}959135$ | $-27{,}992201$ | $-0{,}03307$ |
| Alta frecuencia | $10{,}052201$ | $100$ | $-35{,}978439$ | $-55{,}963290$ | $-20{,}03014$ |

La medición de alta frecuencia se aproxima al valor asintótico de $-20\;\mathrm{dB/década}$, asociado a $Z(s)\sim1/s$.

**Comparación numérica con los datos de simulación incorporados**

Se compara la impedancia simulada con la función obtenida por remociones. Los datos de frecuencia e impedancia utilizados están guardados dentro de este notebook.


In [9]:
# Datos de frecuencia e impedancia incorporados para verificar las simulaciones.
DATOS_SIMULACION = {
  "ejercicio1": "...",
  "ejercicio2": "..."
}


In [10]:
def comparar_ltspice(nombre, funcion, titulo):
    with np.load(BytesIO(base64.b64decode(DATOS_SIMULACION[nombre])), allow_pickle=False) as datos:
        f = datos['frecuencia'].copy()
        z_lt = datos['impedancia'].copy()
    z_teo = sp.lambdify(s, funcion, 'numpy')(2j*np.pi*f)
    error = np.max(np.abs(z_lt-z_teo))
    error_rel = np.max(np.abs((z_lt-z_teo)/z_teo))
    print(f'Puntos comparados: {len(f)}')
    print(f'Error absoluto máximo: {error:.3e} Ω')
    print(f'Error relativo máximo: {error_rel:.3e}')
    print(f'Z a {f[0]:g} Hz: {abs(z_lt[0]):.9g} Ω')
    print(f'Z a {f[-1]:g} Hz: {abs(z_lt[-1]):.9g} Ω')
    assert error_rel < 1e-5
    fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)
    for z,label,style in [(z_teo,'Función teórica','-'),(z_lt,'LTspice','--')]:
        ax[0].semilogx(f,20*np.log10(np.abs(z)),style,label=label)
        ax[1].semilogx(f,np.angle(z,deg=True),style,label=label)
    ax[0].set_title(titulo)
    ax[0].set_ylabel('Módulo de Z (dB ref. 1 Ω)')
    ax[1].set_ylabel('Fase de Z (°)')
    ax[1].set_xlabel('Frecuencia (Hz)')
    for eje in ax:
        eje.grid(True,which='both',alpha=.3)
        eje.legend()
    mostrar_figura(fig, 'TS8_comparacion_'+nombre+'.png', titulo)
    return error, error_rel

errores_ej1 = comparar_ltspice('ejercicio1', Z_ej1, 'Ejercicio 1 - Teoría y LTspice')


Puntos comparados: 15001
Error absoluto máximo: 2.126e-15 Ω
Error relativo máximo: 1.020e-14
Z a 0.001 Hz: 0.0399984588 Ω
Z a 100 Hz: 0.00159160577 Ω


---
## <ins>2. Síntesis del dipolo RC con remociones parciales</ins>

$$Z(s)=\frac{s^2+6s+8}{s^2+4s+3}
=\frac{(s+2)(s+4)}{(s+1)(s+3)}.$$

Los polos son $-1$ y $-3$; los ceros son $-2$ y $-4$. Se intercalan sobre el eje real negativo, en el orden de una impedancia RC:

$$1\;(P)<2\;(C)<3\;(P)<4\;(C),\qquad s=-\sigma.$$

Los bloques $Z_1$, $Z_2$ y $Z_3$ se obtienen como resistencias. Las derivaciones son ramas RC **serie**.

### <ins>Qué fijan las constantes de tiempo</ins>

La admitancia de una rama RC serie es:

$$Y_{RC}(s)=\frac1{R+1/(sC)}
=\frac{(1/R)s}{s+1/(RC)}
=\frac{k s}{s+\sigma},$$

$$k=\frac1R,\qquad\sigma=\frac1{RC},\qquad C=\frac{k}{\sigma}.$$

Así, $R_1C_1=1/5$ exige un polo de admitancia en $s=-5$, y $R_2C_2=1/4$ exige otro en $s=-4$.

Para generar esos polos, primero se remueven parcialmente resistencias serie. Se elige el valor que haga cero al remanente de impedancia en la posición deseada; al invertirlo, ese cero se convierte en polo.

In [11]:
Z_ej2 = (s**2+6*s+8)/(s**2+4*s+3)
mostrar('Z(s)', Z_ej2)
print('Polos:', sp.solve(sp.denom(Z_ej2),s))
print('Ceros:', sp.solve(sp.numer(Z_ej2),s))

<IPython.core.display.Math object>

Polos: [-3, -1]
Ceros: [-4, -2]


### <ins>Primera remoción parcial: resistencia Z1</ins>

La primera derivación debe tener un polo en $-5$. Se requiere que $Z-Z_1$ tenga allí un cero:

$$[Z(s)-Z_1]_{s=-5}=0\quad\Longrightarrow\quad Z_1=Z(-5).$$

$$Z_1=\frac{(-5+2)(-5+4)}{(-5+1)(-5+3)}
=\frac{3}{8}\;\Omega.$$

Se llama remoción parcial porque no se extrae todo el valor en infinito: $Z(\infty)=1$, pero se remueve $3/8$.

$$\begin{aligned}
Z_2&=Z-\frac38\\
&=\frac{8(s^2+6s+8)-3(s^2+4s+3)}{8(s^2+4s+3)}\\
&=\frac{5s^2+36s+55}{8(s+1)(s+3)}\\
&=\boxed{\frac58\frac{(s+11/5)(s+5)}{(s+1)(s+3)}}.
\end{aligned}$$

Al invertir:

$$\boxed{Y_2=\frac85\frac{(s+1)(s+3)}{(s+11/5)(s+5)}}.$$

La remoción desplazó los ceros de $-2$ y $-4$ a $-11/5$ y $-5$.

In [12]:
Z2_ej2, Z1_bloque = remover_valor_en_infinito(Z_ej2, sigma_zero=5)
Y2_ej2 = sp.cancel(1/Z2_ej2)
mostrar('Z_1', Z1_bloque)
mostrar('Z_2(s)', Z2_ej2)
mostrar('Y_2(s)', Y2_ej2)
assert Z1_bloque == sp.Rational(3,8)
verificar_cero(Z2_ej2.subs(s,-5))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### <ins>Segunda remoción: Foster RC de la primera rama serie</ins>

Como la rama está en derivación, se remueve su **admitancia**:

$$Y_{R_1+C_1}(s)=\frac{k_1s}{s+5}.$$

Su coeficiente se obtiene cancelando el polo:

$$\begin{aligned}
k_1&=\lim_{s\to-5}\frac{s+5}{s}Y_2(s)\\
&=\left.\frac85\frac{(s+1)(s+3)}{s(s+11/5)}\right|_{s=-5}\\
&=\frac85\frac{(-4)(-2)}{(-5)(-14/5)}
=\frac{32}{35}.
\end{aligned}$$

De $k_1=1/R_1$ y $C_1=k_1/5$:

$$\boxed{R_1=\frac{35}{32}\;\Omega},\qquad
\boxed{C_1=\frac{32}{175}\;\mathrm{F}}.$$

La constante de tiempo verifica:

$$R_1C_1=\frac{35}{32}\frac{32}{175}=\frac15\;\mathrm{s}.$$

Se resta la rama:

$$\begin{aligned}
Y_4&=Y_2-\frac{32}{35}\frac{s}{s+5}\\
&=\frac{\frac85(s+1)(s+3)-\frac{32}{35}s(s+11/5)}{(s+11/5)(s+5)}\\
&=\frac{\frac{24}{35}(s+7/5)(s+5)}{(s+11/5)(s+5)}\\
&=\boxed{\frac{24}{35}\frac{s+7/5}{s+11/5}}.
\end{aligned}$$

El factor $s+5$ se cancela: la remoción del polo es total. Se invierte para continuar con el siguiente bloque serie:

$$\boxed{Z_4=\frac{35}{24}\frac{s+11/5}{s+7/5}}.$$

En PyTC2: `isImpedance=False, isRC=True`, porque se remueve la admitancia de una rama RC serie.

In [13]:
Y4_ej2, Y_RC1_ej2, R1_ej2, C1_ej2 = remover_polo_sigma(
    Y2_ej2, sigma=5, isImpedance=False, isRC=True
)
Z4_ej2 = sp.cancel(1/Y4_ej2)
mostrar('Y_{R_1+C_1}(s)', Y_RC1_ej2)
mostrar('Y_4(s)', Y4_ej2)
mostrar('Z_4(s)', Z4_ej2)
tabla_componentes([('R1',R1_ej2,'Ω'),('C1',C1_ej2,'F')])
assert R1_ej2*C1_ej2 == sp.Rational(1,5)
verificar_cero(Y_RC1_ej2-1/(R1_ej2+1/(s*C1_ej2)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

R1: 35/32 = 1.09375 Ω
C1: 32/175 = 0.182857143 F


### <ins>Tercera remoción parcial: resistencia Z2</ins>

La segunda rama exige un polo de admitancia en $-4$. Se elige la resistencia serie que genere un cero de impedancia allí:

$$Z_2\text{ (bloque)}=Z_4(-4)
=\frac{35}{24}\frac{-4+11/5}{-4+7/5}
=\frac{35}{24}\frac9{13}
=\boxed{\frac{105}{104}\;\Omega}.$$

El bloque $Z_2$ del dibujo es una resistencia; no debe confundirse con el remanente $Z_2(s)$ del primer paso.

$$\begin{aligned}
Z_6&=Z_4-\frac{105}{104}\\
&=\frac{\frac{35}{24}(s+11/5)-\frac{105}{104}(s+7/5)}{s+7/5}\\
&=\boxed{\frac{35}{78}\frac{s+4}{s+7/5}}.
\end{aligned}$$

Se invierte:

$$\boxed{Y_6=\frac{78}{35}\frac{s+7/5}{s+4}}.$$

El polo en $-4$ ya está disponible para remover la segunda rama RC.

In [14]:
Z6_ej2, Z2_bloque = remover_valor_en_infinito(Z4_ej2, sigma_zero=4)
Y6_ej2 = sp.cancel(1/Z6_ej2)
mostrar(r'Z_2\mathrm{\ (bloque)}', Z2_bloque)
mostrar('Z_6(s)', Z6_ej2)
mostrar('Y_6(s)', Y6_ej2)
assert Z2_bloque == sp.Rational(105,104)
verificar_cero(Z6_ej2.subs(s,-4))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### <ins>Cuarta remoción: Foster RC de la segunda rama serie</ins>

Se remueve:

$$Y_{R_2+C_2}(s)=\frac{k_2s}{s+4}.$$

El coeficiente es:

$$\begin{aligned}
k_2&=\lim_{s\to-4}\frac{s+4}{s}Y_6(s)\\
&=\left.\frac{78}{35}\frac{s+7/5}{s}\right|_{s=-4}\\
&=\frac{78}{35}\frac{-13/5}{-4}
=\boxed{\frac{507}{350}}.
\end{aligned}$$

Por identificación:

$$\boxed{R_2=\frac{350}{507}\;\Omega},\qquad
\boxed{C_2=\frac{k_2}{4}=\frac{507}{1400}\;\mathrm{F}}.$$

Se comprueba:

$$R_2C_2=\frac{350}{507}\frac{507}{1400}=\frac14\;\mathrm{s}.$$

La resta deja una conductancia constante:

$$\begin{aligned}
Y_8&=\frac{78}{35}\frac{s+7/5}{s+4}
-\frac{507}{350}\frac{s}{s+4}\\
&=\frac{\frac{39}{50}s+\frac{78}{25}}{s+4}
=\frac{\frac{39}{50}(s+4)}{s+4}
=\boxed{\frac{39}{50}}.
\end{aligned}$$

### <ins>Quinta remoción: resistencia final Z3</ins>

Al invertir el último resto:

$$\boxed{Z_3=Z_8=\frac1{Y_8}=\frac{50}{39}\;\Omega}.$$

La extracción de esta resistencia termina la síntesis.

In [15]:
Y8_ej2, Y_RC2_ej2, R2_ej2, C2_ej2 = remover_polo_sigma(
    Y6_ej2, sigma=4, isImpedance=False, isRC=True
)
Z8_ej2 = sp.cancel(1/Y8_ej2)
Z3_bloque = Z8_ej2
mostrar('Y_{R_2+C_2}(s)', Y_RC2_ej2)
mostrar('Y_8(s)', Y8_ej2)
mostrar('Z_3', Z3_bloque)
tabla_componentes([('R2',R2_ej2,'Ω'),('C2',C2_ej2,'F')])
assert R2_ej2*C2_ej2 == sp.Rational(1,4)
assert Z3_bloque == sp.Rational(50,39)
verificar_cero(Y_RC2_ej2-1/(R2_ej2+1/(s*C2_ej2)))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

R2: 350/507 = 0.690335306 Ω
C2: 507/1400 = 0.362142857 F


### <ins>Mapa de remociones</ins>

<div align="center">
    <img src="./Imagenes/TS8_mapa_remociones_2.png" alt="Mapa de remociones con alturas proporcionales en cero e infinito" width="700" />
</div>

### <ins>Valores de los componentes</ins>

| Componente | Valor exacto | Aproximación | Conexión |
|---|---|---:|---|
| $Z_1$ | $3/8\;\Omega$ | $0{,}375\;\Omega$ | Resistencia serie |
| $Z_2$ | $105/104\;\Omega$ | $1{,}009615\;\Omega$ | Resistencia serie |
| $Z_3$ | $50/39\;\Omega$ | $1{,}282051\;\Omega$ | Resistencia final |
| $R_1$ | $35/32\;\Omega$ | $1{,}09375\;\Omega$ | Primera rama RC serie |
| $C_1$ | $32/175\;\mathrm{F}$ | $0{,}182857\;\mathrm{F}$ | Primera rama RC serie |
| $R_2$ | $350/507\;\Omega$ | $0{,}690335\;\Omega$ | Segunda rama RC serie |
| $C_2$ | $507/1400\;\mathrm{F}$ | $0{,}362143\;\mathrm{F}$ | Segunda rama RC serie |

### <ins>Circuito obtenido</ins>

<div align="center">
    <img src="./Imagenes/TS8_circuito_2.png" alt="Circuito RC obtenido con los valores exactos" width="700" />
</div>

### <ins>Reconstrucción y verificación simbólica</ins>

Desde el extremo derecho:

$$Z_{RC,1}=R_1+\frac1{sC_1},\qquad Z_{RC,2}=R_2+\frac1{sC_2}.$$

La segunda rama está en paralelo con $Z_3$:

$$Z_{\mathrm{derecha}}=\left(\frac1{Z_{RC,2}}+\frac1{Z_3}\right)^{-1}.$$

Se suma $Z_2$ y se conecta en paralelo con la primera rama:

$$Z_{\mathrm{red}}=Z_1+
\left[\frac1{Z_{RC,1}}+\frac1{Z_2+Z_{\mathrm{derecha}}}\right]^{-1}.$$

Se verifica la igualdad exacta con la función propuesta y las dos constantes de tiempo.

In [16]:
Z_RC1_red = R1_ej2+1/(s*C1_ej2)
Z_RC2_red = R2_ej2+1/(s*C2_ej2)
Z_derecha = 1/(1/Z_RC2_red+1/Z3_bloque)
Z_red_ej2 = sp.cancel(Z1_bloque+1/(1/Z_RC1_red+1/(Z2_bloque+Z_derecha)))
mostrar('Z_{red}(s)', Z_red_ej2)
mostrar('Z_{red}(s)-Z(s)', sp.cancel(Z_red_ej2-Z_ej2))
verificar_cero(Z_red_ej2-Z_ej2)
mostrar('R_1C_1', R1_ej2*C1_ej2)
mostrar('R_2C_2', R2_ej2*C2_ej2)
assert all(x>0 for x in [Z1_bloque,Z2_bloque,Z3_bloque,R1_ej2,C1_ej2,R2_ej2,C2_ej2])

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### <ins>Comportamiento en frecuencia</ins>

En DC, los capacitores abren las dos derivaciones. La entrada ve las tres resistencias serie:

$$Z(0)=Z_1+Z_2+Z_3=\frac38+\frac{105}{104}+\frac{50}{39}=\frac83\;\Omega.$$

A alta frecuencia, los capacitores cortocircuitan dentro de sus ramas y quedan redes resistivas:

$$Z(\infty)=Z_1+\left[R_1\parallel\left(Z_2+(R_2\parallel Z_3)\right)\right]=1\;\Omega.$$

El módulo pasa de $20\log_{10}(8/3)=8{,}5194\;\mathrm{dB}$ a $0\;\mathrm{dB}$ y la fase tiende a $0^\circ$ en ambos extremos.

Los polos y ceros son reales negativos: **las frecuencias de quiebre no son picos ni nulos del barrido AC**. Las frecuencias asociadas a los factores de primer orden son:

| Singularidad | Posición en el plano s | Frecuencia de quiebre (Hz) |
|---|---|---:|
| Polo 1 | $-1$ | $1/(2\pi)=0{,}159155$ |
| Cero 1 | $-2$ | $2/(2\pi)=0{,}318310$ |
| Polo 2 | $-3$ | $3/(2\pi)=0{,}477465$ |
| Cero 2 | $-4$ | $4/(2\pi)=0{,}636620$ |

Las pendientes asintóticas ideales por factores son $0$, $-20$, $0$, $-20$ y $0\;\mathrm{dB/década}$. Como los quiebres están próximos entre sí, los tramos intermedios no aparecen como rectas largas.

In [17]:
mostrar('Z(0)', sp.limit(Z_ej2,s,0))
mostrar(r'Z(\infty)', sp.limit(Z_ej2,s,sp.oo))
assert sp.limit(Z_red_ej2,s,0)==sp.Rational(8,3)
assert sp.limit(Z_red_ej2,s,sp.oo)==1
bode(Z_ej2, 'Ejercicio 2 - Impedancia RC teórica')

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### <ins>Simulación en LTspice</ins>

**Circuito simulado**

<div align="center">
    <img src="./Imagenes/Ejercicio2 - Simulacion - Circuito .png" alt="Circuito del ejercicio 2 en LTspice" width="700" />
</div>

El circuito utiliza los valores exactos obtenidos por remociones. La impedancia se mide mediante `V(in)/(-I(V1))`.

**Módulo y fase**

<div align="center">
    <img src="./Imagenes/Ejercicio1 - Simulacion - Modulo y Fase .png" alt="Módulo y fase del ejercicio 2" width="1000" />
</div>

El módulo desciende desde aproximadamente $8{,}52\;\mathrm{dB}$ hasta $0\;\mathrm{dB}$, de acuerdo con $Z(0)=8/3\;\Omega$ y $Z(\infty)=1\;\Omega$. La fase es negativa en la zona intermedia y se aproxima a $0^\circ$ en los extremos, como corresponde al dipolo RC.

**Cursores próximos a los quiebres de polos y ceros**

<div align="center">
    <img src="./Imagenes/Ejercicio1 - Simulacion- Polo y cero parte 1.png" alt="Cursores próximos al primer polo y al primer cero del ejercicio 2" width="300" />
</div>

<div align="center">
    <img src="./Imagenes/Ejercicio1 - Simulacion- Polo y cero parte 2.png" alt="Cursores próximos al segundo polo y al segundo cero del ejercicio 2" width="300" />
</div>

| Singularidad | Frecuencia teórica (Hz) | Frecuencia del cursor (Hz) | Módulo leído (dB) | Fase leída (°) |
|---|---:|---:|---:|---:|
| Polo en $-1$ | $0{,}159155$ | $0{,}15922087$ | $6{,}2826618$ | $-22{,}837562$ |
| Cero en $-2$ | $0{,}318310$ | $0{,}31841975$ | $3{,}9108443$ | $-25{,}559204$ |
| Polo en $-3$ | $0{,}477465$ | $0{,}47752927$ | $2{,}5657010$ | $-23{,}384126$ |
| Cero en $-4$ | $0{,}636620$ | $0{,}63679552$ | $1{,}7772364$ | $-20{,}655989$ |

Las frecuencias de los cursores están próximas a $f=\sigma/(2\pi)$ de cada factor. Los valores de módulo y fase son lecturas de la **impedancia total**, por lo que no deben compararse con los $\pm3{,}0103\;\mathrm{dB}$ de un factor normalizado aislado. Las posiciones exactas de las singularidades se verifican mediante la factorización de $Z(s)$.

**Comparación numérica con los datos de simulación incorporados**

Se verifica la función completa con los datos de frecuencia e impedancia guardados en el notebook.


In [18]:
errores_ej2 = comparar_ltspice('ejercicio2', Z_ej2, 'Ejercicio 2 - Teoría y LTspice')

Puntos comparados: 15001
Error absoluto máximo: 7.116e-15 Ω
Error relativo máximo: 2.671e-15
Z a 0.001 Hz: 2.66662463 Ω
Z a 100 Hz: 1.00001266 Ω


---
# <ins>Resolución manuscrita</ins>

El desarrollo a mano se muestra en el visor siguiente.

In [4]:
from IPython.display import IFrame
IFrame("Imagenes/TS8.pdf", width=1000, height=700)